<a href="https://colab.research.google.com/github/Haross/sql_notebook/blob/main/SQL_challenge_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL In-Class Challenge – All Topics

Today you will work as **SQL analysts** for a language-learning startup.

Ruby is the creator of a **language tandem app** — a platform where people learning languages can connect with native speakers and practice together through chat.

Ruby now wants to better understand how the app is being used so she can improve the platform and grow the business.

She has asked you to analyze the app’s database and answer important questions such as:

- How many countries have Spanish speakers?
- How many chatrooms are active?
- Which speakers could be good language-exchange partners?

The answers are hidden in the database — and **your SQL skills will uncover them**.

---
## Challenge format

There are **11 SQL challenges** available.

- You may solve the exercises **in any order**.
- Each correct solution gives **points for the leaderboard** depending on its difficulty.
- The **final leaderboard will be shown at the end of class**.
- 🏆 The top student will receive a **small bonus**.

⚠️ **Exercise 11 is also the graded exercise for this activity**, so make sure you complete it.

Your goal: **solve as many challenges as you can and climb the leaderboard.**


## **SQL Environment Setup (do not edit)**

In [20]:
# @title

%%capture
!mkdir -p notebook_lib
!wget --cache=off -q -O notebook_lib/sql_runner.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner.py
!wget --cache=off -q -O notebook_lib/sql_runner_store.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner_store.py
!wget --cache=off -q -O notebook_lib/sql_runner_ui_bits.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner_ui_bits.py
!wget --cache=off -q -O notebook_lib/validators.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/validators.py

!wget --cache=off -q -O notebook_lib/cloud_submitter.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/cloud_submitter.py

!touch notebook_lib/__init__.py
#!pip install -q duckdb

from notebook_lib.sql_runner import make_sql_runner
from notebook_lib.validators import make_df_validator_nospoilers, check_process_rules
from notebook_lib.cloud_submitter import make_cloud_run_submitter
import duckdb
import pandas as pd
from pathlib import Path


In [21]:
# @title

DB_FILE = 'class_joins.duckdb'

if DB_FILE != ":memory:":
    Path(DB_FILE).unlink(missing_ok=True)

conn = duckdb.connect(DB_FILE)

conn.execute(r'''
-- DuckDB schema + seed data

DROP TABLE IF EXISTS chatroom_speaker;
DROP TABLE IF EXISTS language_speaker;
DROP TABLE IF EXISTS chatroom;
DROP TABLE IF EXISTS speaker;
DROP TABLE IF EXISTS language;

CREATE TABLE speaker (
    id INTEGER PRIMARY KEY,
    name VARCHAR NOT NULL,
    email VARCHAR NOT NULL,
    country VARCHAR NOT NULL,
    city VARCHAR NOT NULL
);

CREATE TABLE language (
    id INTEGER PRIMARY KEY,
    name VARCHAR NOT NULL
);

CREATE TABLE chatroom (
    id INTEGER PRIMARY KEY,
    created_at TIMESTAMP NOT NULL,
    last_active_at TIMESTAMP NOT NULL
);

CREATE TABLE language_speaker (
    speaker_id INTEGER NOT NULL,
    language_id INTEGER NOT NULL,
    native BOOLEAN NOT NULL,
    PRIMARY KEY (speaker_id, language_id),
    FOREIGN KEY (speaker_id) REFERENCES speaker(id),
    FOREIGN KEY (language_id) REFERENCES language(id)
);

CREATE TABLE chatroom_speaker (
    speaker_id INTEGER NOT NULL,
    chatroom_id INTEGER NOT NULL,
    PRIMARY KEY (speaker_id, chatroom_id),
    FOREIGN KEY (speaker_id) REFERENCES speaker(id),
    FOREIGN KEY (chatroom_id) REFERENCES chatroom(id)
);

INSERT INTO speaker (id, name, email, country, city) VALUES
    (1, 'Karli Roberson', 'karli.roberson@example.com', 'USA', 'New York'),
    (2, 'Emma Dumont', 'emma.dumont@example.com', 'USA', 'New York'),
    (3, 'Madeleine Renaud', 'm.renaud@example.com', 'USA', 'New York'),
    (4, 'Anna Kowalska', 'a.kowalska@example.com', 'Poland', 'Warsaw'),
    (5, 'Alba Rodriguez', 'a.rodriguez@example.com', 'Spain', 'Barcelona'),
    (6, 'Gabriel Navarra', 'gabriel.nav@example.com', 'Spain', 'Barcelona'),
    (7, 'Beatrice Fabri', 'b.fabri@example.com', 'Poland', 'Warsaw'),
    (8, 'Lina Pereira', 'pereira@example.com', 'Poland', 'Warsaw'),
    (9, 'Giovanni Ferretti', 'feretti@example.com', 'Spain', 'Madrid'),
    (10, 'Maya Martinazzo', 'maya.m@example.com', 'Italy', 'Naples'),
    (11, 'Erika Weber', 'eweber@example.com', 'Italy', 'Naples'),
    (12, 'Constance Deparrois', 'c.depa@example.com', 'France', 'Paris'),
    (13, 'Julia Nowak', 'rverano@example.com', 'France', 'Paris'),
    (14, 'Susan Gray', 'susan.gray@example.com', 'France', 'Paris'),
    (15, 'Patrick Turner', 'pat.turner@example.com', 'France', 'Paris');

INSERT INTO language (id, name) VALUES
    (1, 'English'),
    (2, 'Polish'),
    (3, 'Spanish'),
    (4, 'French'),
    (5, 'German'),
    (6, 'Italian');

INSERT INTO chatroom (id, created_at, last_active_at) VALUES
    (1, '2015-01-10 00:55:58', '2020-05-25 07:46:00'),
    (2, '2020-02-11 09:07:12', '2022-06-01 16:55:29'),
    (3, '2020-02-10 21:12:52', '2020-08-27 19:56:03'),
    (4, '2019-03-31 18:28:18', '2022-02-11 14:40:15'),
    (5, '2017-04-18 21:58:36', '2021-06-17 23:49:00'),
    (6, '2018-03-01 09:00:34', '2018-11-18 18:45:04'),
    (7, '2019-10-14 17:35:12', '2020-09-19 16:36:59'),
    (8, '2020-06-13 12:32:56', '2021-01-07 14:15:02'),
    (9, '2020-01-20 00:05:19', '2020-12-22 01:19:16'),
    (10, '2022-01-03 02:00:04', '2022-03-01 11:59:00');

INSERT INTO language_speaker (speaker_id, language_id, native) VALUES
    (1, 1, TRUE),
    (2, 1, FALSE),
    (2, 5, FALSE),
    (2, 4, TRUE),
    (3, 4, TRUE),
    (3, 1, TRUE),
    (3, 5, FALSE),
    (4, 2, TRUE),
    (4, 1, FALSE),
    (4, 4, FALSE),
    (4, 5, FALSE),
    (5, 3, TRUE),
    (5, 1, FALSE),
    (5, 6, FALSE),
    (6, 3, TRUE),
    (6, 1, FALSE),
    (7, 6, TRUE),
    (7, 2, TRUE),
    (7, 1, FALSE),
    (7, 4, FALSE),
    (9, 6, TRUE),
    (9, 3, FALSE),
    (10, 6, TRUE),
    (10, 1, FALSE),
    (11, 5, TRUE),
    (11, 6, FALSE),
    (12, 4, TRUE),
    (13, 2, TRUE),
    (13, 4, TRUE),
    (13, 1, FALSE),
    (14, 1, TRUE),
    (14, 4, FALSE),
    (15, 1, TRUE),
    (15, 4, TRUE),
    (15, 3, FALSE);

INSERT INTO chatroom_speaker (speaker_id, chatroom_id) VALUES
    (5, 1),
    (9, 1),
    (5, 2),
    (15, 2),
    (4, 3),
    (15, 3),
    (7, 4),
    (15, 4),
    (2, 5),
    (14, 5),
    (6, 6),
    (15, 6),
    (13, 7),
    (14, 7),
    (3, 8),
    (4, 8),
    (3, 9),
    (7, 9),
    (2, 10),
    (7, 10);
''')
print(f"Duckdb database ready ✅ ({DB_FILE})")


Duckdb database ready ✅ (class_joins.duckdb)


In [22]:
# @title Cloud submission config (do not edit)
CLOUD_SUBMIT_URL = 'https://cinemax-validator-830800716261.europe-west1.run.app/submit'
CLOUD_EXAM_ID = 'SQL_CHALLENGE_1_2026'
CLOUD_API_KEY = 'SECRET123'


In [31]:
# @title
import ipywidgets as widgets
from IPython.display import display, HTML
from pathlib import Path

TOKEN_FILE = Path("student_token.txt")

status_out = widgets.Output()

def update_status(title, message, type="info"):
    icons = {
        "info": "ℹ️",
        "success": "✅",
        "warning": "⚠️",
        "error": "❌",
    }

    icon = icons.get(type, "ℹ️")

    with status_out:
        status_out.clear_output()
        display(HTML(f"""
        <style>
        /* ===== Theme Variables ===== */
        html {{
            --info-border: #1a73e8;
            --info-bg: #eef3fe;

            --success-border: #2e7d32;
            --success-bg: #eaf7ee;

            --warning-border: #f9a825;
            --warning-bg: #fff8e1;

            --error-border: #c62828;
            --error-bg: #fdecea;

            --text-color: #202124;
            --shadow: 0 2px 6px rgba(0,0,0,0.05);
        }}

        html[theme="dark"] {{
            --info-border: #8ab4f8;
            --info-bg: #1e2a3a;

            --success-border: #81c995;
            --success-bg: #1f3324;

            --warning-border: #fdd663;
            --warning-bg: #3a3218;

            --error-border: #f28b82;
            --error-bg: #3a1f1f;

            --text-color: #e8eaed;
            --shadow: 0 2px 8px rgba(0,0,0,0.6);
        }}

        /* ===== Status Box ===== */
        .status-box {{
            padding:16px;
            border-left:6px solid var(--{type}-border);
            background: var(--{type}-bg);
            border-radius:10px;
            font-family:Arial, sans-serif;
            line-height:1.6;
            box-shadow: var(--shadow);
            color: var(--text-color);
        }}

        .status-title {{
            font-size:16px;
            font-weight:600;
            margin-bottom:6px;
        }}

        .status-message {{
            font-size:14px;
        }}
        </style>

        <div class="status-box">
            <div class="status-title">
                {icon} {title}
            </div>
            <div class="status-message">
                {message}
            </div>
        </div>
        """))

display(status_out)


t = TOKEN_FILE.read_text().strip() if TOKEN_FILE.exists() else ""

update_status(
    "Important: Before Starting",
    """
    The next cell will ask you to enter your <b>student token</b>.<br><br>
    Your token is <b>your first name + the first letter of your last name</b>,
    all in <b>lowercase</b>
    After typing it, press <b>Enter</b>.<br>
    You only need to do this once.
    """,
    type="info"
)

Output()

In [ ]:
# @title
from pathlib import Path

TOKEN_FILE = Path("student_token.txt")

existing = TOKEN_FILE.read_text().strip() if TOKEN_FILE.exists() else ""
if not existing:
    while True:
        token = input("Student token: ").strip()
        if token:
            break
        print("⚠️ Token cannot be empty. Try again.")

    TOKEN_FILE.write_text(token)
    update_status(
        "Token Saved Successfully",
        f"Your token has been stored securely.<br><br>Your token is:<b>{token}</b>",
        type="success"
    )
    print("✅ Token saved.")
else:
    token = existing
    update_status(
        "Token Already Saved",
        f"Your token has been already stored securely.<br><br>Your token is: <b>{token}</b>",
        type="success"
    )
    print("✅ Token already saved.")

## Get to know the tables

### The database and the `speaker` table

The app database consists of five tables:

- `speaker`
- `language`
- `language_speaker`
- `chatroom`
- `chatroom_speaker`

Let’s take a closer look at each of them.

### `speaker`

The first table, `speaker`, contains information about the users of the app.

It has the following columns:

- `id` – the unique identifier of each speaker  
- `name` – the speaker’s name  
- `email` – the speaker’s email address  
- `country` – the country where the speaker is located  
- `city` – the city where the speaker is located  

---

### The `language` table

The table `language` stores the languages available in the app.

It contains the following columns:

- `id` – the unique identifier of each language  
- `name` – the name of the language  

---

### The `language_speaker` table

The table `language_speaker` connects speakers with languages.

It contains the following columns:

- `speaker_id` – the id of the speaker  
- `language_id` – the id of the language  
- `native` – a Boolean value indicating whether the speaker is a **native speaker** of that language or a **learner**

This table allows one speaker to be linked to multiple languages.

---

### The `chatroom` table

The table `chatroom` contains information about the conversations.

It has the following columns:

- `id` – the unique identifier of the chatroom  
- `created_at` – the timestamp when the chatroom was created  
- `last_active_at` – the timestamp of the most recent message sent in the chatroom  

Note that each chatroom can contain **exactly two speakers**. Group chats are not allowed.

---

### The `chatroom_speaker` table

Finally, the table `chatroom_speaker` connects speakers with chatrooms.

It contains the following columns:

- `speaker_id` – the id of the speaker  
- `chatroom_id` – the id of the chatroom  

This table tells us which two speakers participate in each chatroom.


In [25]:
# @title ER diagram
%%html
<img id="er-img" style="width:67%; max-width:100%;  height:auto;"
     data-light="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/ER_sql_challenge_1_horizontal.png"
     data-dark="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/ER_sql_challenge_1_horizontal_black.png"
     alt="ER diagram">

<script>
  const img = document.getElementById("er-img");

  function isDarkTheme() {
    // Colab sets html[theme=dark] on the top document
    const themeAttr = document.documentElement.getAttribute("theme");
    if (themeAttr) return themeAttr === "dark";

    // fallback: OS/browser preference
    return window.matchMedia && window.matchMedia("(prefers-color-scheme: dark)").matches;
  }

  function updateImage() {
    img.src = isDarkTheme() ? img.dataset.dark : img.dataset.light;
  }

  updateImage();

  // React to Colab theme toggles (attribute changes)
  new MutationObserver(updateImage).observe(document.documentElement, {
    attributes: true,
    attributeFilter: ["theme"]
  });

  // React to OS/browser theme changes (fallback)
  if (window.matchMedia) {
    const mq = window.matchMedia("(prefers-color-scheme: dark)");
    mq.addEventListener?.("change", updateImage);
    mq.addListener?.(updateImage); // older browsers
  }
</script>

In [26]:
# @title In-class exercise 1 (5 points)
submitter_in_class_ex_1 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='ex1',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="in_class_ex_1",
    description_md='### In-class exercise 1 (5 points - Easy)\nRuby would like to promote the app in Paris. Some potential users may be interested in finding speaking partners with whom they can talk not only online but also in person. So, she wants to show in her campaign all the languages that can be practiced with native speakers located in Paris. Can you create such a list for her?\n\nDisplay a list of language names. There should be only one column: name.\n\nInclude native languages only of the speakers located in Paris. Each language should appear on the list just once.          \n',
    submitter=submitter_in_class_ex_1,
    sol_sql=None,
    select_only=True,
    dedupe=True
)


In [27]:
# @title In-class exercise 10 (5 points)
submitter_in_class_ex_10 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='ex10',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="in_class_ex_10",
    description_md='### In-class exercise 10 (5 points - Easy)\nRuby wants to analyze chatroom activity based on when chatrooms were created and when they were last active.\n\nDisplay the chatroom **id**, **created_at**, and **last_active_at**.\n\nReturn chatrooms that meet the following criteria:\n\n* Chatrooms **created before 2019** that have been **active in 2020 or later**\n\nAlso include chatrooms that:\n\n* were **created in 2020 or later**\n* were **last active before 2021** or **last active in 2022 or later**\n\nSort the results by **id**.\n',
    submitter=submitter_in_class_ex_10,
    sol_sql=None,
    select_only=True,
    dedupe=True
)


In [28]:
# @title In-class exercise 4 (10 points)
submitter_in_class_ex_4 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='ex4',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="in_class_ex_4",
    description_md="### In-class exercise 4 (10 points - Medium)\nRuby would like to know how many languages are being studied by each user. Do you know how to find this information in the database?\n\nShow the speaker's name and the count of non-native languages spoken by each speaker (call this column language_count). Some speakers may not yet be assigned to any language; display 0 instead of omitting that speaker. Sort the results by the number of languages in descending order.\n",
    submitter=submitter_in_class_ex_4,
    sol_sql=None,
    select_only=True,
    dedupe=True
)


In [29]:
# @title In-class exercise 8 (5 points)
submitter_in_class_ex_8 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='ex8',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="in_class_ex_8",
    description_md='### In-class exercise 8 (5 points - Medium)\nRuby is curious whether users in the same city might want to meet in person. She asks you to find **pairs of speakers who live in the same city**.\n\nTo avoid duplicates, each pair should appear **only once**, and the first speaker in the pair must have a **smaller ID**. Display three columns:\n\n* speaker1 – name of the first speaker\n* speaker2 – name of the second speaker\n* city – the city both speakers live in\n\n### Conditions\n\n* Both speakers must live in the **same city**\n* speaker1.id < speaker2.id\n* Sort results by city, then speaker1\n',
    submitter=submitter_in_class_ex_8,
    sol_sql=None,
    select_only=True,
    dedupe=True
)


In [30]:
# @title In-class exercise 7 (15 points)
submitter_in_class_ex_7 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='ex7',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="in_class_ex_7",
    description_md="### In-class exercise 7 (15 points - Hard)\nRuby would like to show recommendations to the app users. These recommendations will include other speakers with whom the user may be interested in chatting. She has asked you to find all speaker pairs in which one person may learn from the other (i.e., they speak at least one common language, and one is a native speaker of the language while the other is learning it). Can you prepare a list of such pairs?\n\nDisplay two columns: speaker1 and speaker2. Both columns should contain the speakers' names, and together they should form pairs of speakers such that one is a native speaker of a language and the other is studying it. For example, Ann is a native English speaker learning French, and Hugo is a native Spanish speaker learning English; they make a pair for studying English. The ID of speaker1 should be smaller than the ID of speaker2.\n",
    submitter=submitter_in_class_ex_7,
    sol_sql=None,
    select_only=True,
    dedupe=True
)
